In [1]:
from pathlib import Path
import pandas as pd

In [2]:
OUT = Path("/.../data/output")

In [34]:
def driver_analysis():
    actual = pd.read_csv(OUT/"fact_sales_actual.csv", parse_dates=["date"])
    budget = pd.read_csv(OUT/"fact_sales_budget.csv", parse_dates=["date"])
    
    joint_column = ["date", "region_id", "product_id"]
    actual = actual.groupby(joint_column).agg(
        actual_volume = ("volume", "sum"),
        actual_price = ("price", "mean"),
        actual_revenue = ("revenue", "sum")).reset_index()
    budget = budget.groupby(joint_column).agg(
        budget_volume = ("volume", "sum"),
        budget_price = ("price", "mean"),
        budget_revenue = ("revenue", "sum")).reset_index()
    r = actual.merge(budget, on=joint_column)
    
    r["volume_effect"] = (r.actual_volume - r.budget_volume) * r.budget_price
    r["price_effect"] = r.budget_volume * (r.actual_price - r.budget_price)
    r["revenue_variance"] = r.actual_revenue - r.budget_revenue
    r["unexplained_effect"] = r.revenue_variance - r.volume_effect - r.price_effect
    
    long = r.melt(
        id_vars = joint_column,
        value_vars = ["volume_effect", "price_effect", "unexplained_effect"],
        var_name = "driver",
        value_name = "impact")
    
    long2 = long.copy()
    
    long = long.to_csv(OUT/f"driver_analysis.csv", index = False)
    
    print(long2)
    
    return long

In [35]:
driver_analysis()

           date region_id product_id              driver  impact
0    2024-01-01      APAC        P01       volume_effect     0.0
1    2024-01-01      APAC        P02       volume_effect     0.0
2    2024-01-01      APAC        P03       volume_effect     0.0
3    2024-01-01      APAC        P04       volume_effect     0.0
4    2024-01-01        EU        P01       volume_effect     0.0
...         ...       ...        ...                 ...     ...
1291 2026-12-01        EU        P04  unexplained_effect     0.0
1292 2026-12-01     OTHER        P01  unexplained_effect     0.0
1293 2026-12-01     OTHER        P02  unexplained_effect     0.0
1294 2026-12-01     OTHER        P03  unexplained_effect     0.0
1295 2026-12-01     OTHER        P04  unexplained_effect     0.0

[1296 rows x 5 columns]
